# Statistiques descriptives des marchés publics

Suite de `01-exploration-decp.ipynb`. Le premier notebook répondait à « ces données sont-elles
utilisables ». Celui-ci répond à « que disent-elles ».

**Périmètre** : l'état actuel de chaque marché, soit **1 833 468 marchés**. L'historique des
avenants est écarté, sinon un même marché serait compté plusieurs fois.

**Ce qui en sort** : les agrégats sont écrits en CSV par `mesures/decp_distributions.py`, et les
figures du mémoire sont tracées depuis ces CSV par `mesures/figures.py`. Aucun chiffre du mémoire
n'est saisi à la main, et aucune figure ne relit le fichier de 236 Mo.

## Pourquoi cette étape, et pourquoi maintenant

La tentation serait de passer directement à la détection des marchés suspects. Ce serait une faute
de méthode : un modèle entraîné sur des données dont on ignore la forme apprend surtout leurs
défauts. Deux exemples déjà rencontrés dans ce jeu :

- les montants aberrants faussent toute moyenne, donc tout écart à la moyenne ;
- le nombre d'offres reçues manque une fois sur deux, donc l'indicateur de risque principal ne
  couvre que la moitié du terrain.

C'est aussi ce que raconte l'échec de **Google Flu Trends** : un modèle statistique appliqué à des
données dont personne n'avait interrogé la stabilité, et qui a dérivé en silence.

In [1]:
import duckdb
import pandas as pd

con = duckdb.connect()
con.register("decp", con.read_parquet("../donnees/brut/decp.parquet"))

# Une vue plutôt qu'un filtre répété : le périmètre est défini une seule fois, visible en un
# coup d'œil, et impossible à oublier dans une requête suivante.
con.execute("create or replace view marches as select * from decp where donneesActuelles")

nombre_de_marches = con.sql("select count(*) from marches").fetchone()[0]
print(f"{nombre_de_marches:,} marchés".replace(",", " "))

2 114 675 marchés


## 1. Les montants : une masse de petits marchés

La médiane est bien plus parlante que la moyenne sur ces données : une seule ligne à 100 milliards
d'euros déplace la moyenne, pas la médiane. C'est la première règle de lecture d'un jeu pollué par
des valeurs aberrantes.

In [2]:
con.sql("""
    select
        round(quantile_cont(montant, 0.10), 0) as decile_1,
        round(quantile_cont(montant, 0.25), 0) as quartile_1,
        round(median(montant), 0)              as mediane,
        round(quantile_cont(montant, 0.75), 0) as quartile_3,
        round(quantile_cont(montant, 0.90), 0) as decile_9,
        round(quantile_cont(montant, 0.99), 0) as centile_99
    from marches
    where montant > 0
""").df()

,decile_1,quartile_1,mediane,quartile_3,decile_9,centile_99
0,18000.0,50000.0,126372.0,400000.0,1527023.0,23470565.0


**Lecture** : la moitié des marchés sont inférieurs à environ 126 000 euros, et un sur dix dépasse
1,5 million. Autrement dit, la commande publique est faite d'un très grand nombre de petits
marchés, et d'une poignée de très gros qui portent l'essentiel des montants. Toute analyse qui
raisonne en moyenne écrase cette réalité.

Les seuils utilisés dans la figure ne sont pas arbitraires : **25 000, 90 000 et 214 000 euros**
sont des seuils de procédure du code de la commande publique. Découper selon les règles du domaine
plutôt que selon des tranches rondes rend la figure directement interprétable par un acheteur.

In [3]:
# Les agrégats sont déjà calculés et enregistrés par mesures/decp_distributions.py.
# On les relit ici plutôt que de refaire le calcul : c'est exactement ce que fera la figure.
pd.read_csv("../mesures/resultats/montants-tranches.csv")

,tranche,marches,part_pourcent
0,1. moins de 25 k,261300,12.8
1,2. 25 a 90 k,573424,28.0
2,3. 90 a 214 k,459896,22.5
3,4. 214 k a 1 M,470128,23.0
4,5. 1 a 10 M,235530,11.5
5,6. plus de 10 M,47442,2.3


## 2. La découverte principale : le total annuel est inutilisable tel quel

La commande publique française représente environ **230 milliards d'euros par an**. Or la somme
brute des montants du fichier donne **3 030 milliards pour 2020** et **4 725 milliards pour 2025**.

Un facteur d'erreur de dix à vingt. Il ne s'agit pas d'une subtilité méthodologique : le chiffre
brut est simplement faux, et quiconque le publierait sans le corriger se ferait reprendre.

In [4]:
con.sql("""
    select
        year(dateNotification)       as annee,
        count(*)                     as marches,
        round(sum(montant) / 1e9, 1) as total_brut_milliards,
        round(sum(case when montant_anomalie is null then montant else 0 end) / 1e9, 1)
            as total_nettoye_milliards
    from marches
    where dateNotification between date '2019-01-01' and date '2025-12-31'
    group by 1
    order by 1
""").df()

,annee,marches,total_brut_milliards,total_nettoye_milliards
0,2019,141903,267.6,57.0
1,2020,191183,3030.2,113.3
2,2021,261285,3157.9,243.7
3,2022,287476,2511.3,196.4
4,2023,296356,1988.2,256.5
5,2024,322112,2071.0,236.2
6,2025,343756,4724.8,258.3


**Le résultat est net** : en écartant les seules lignes que le producteur signale déjà comme
suspectes ou aberrantes, les totaux retombent entre 196 et 258 milliards pour 2022 à 2025, c'est-à-dire
dans l'ordre de grandeur attendu.

Deux enseignements pour la suite :

1. **Le nettoyage n'est pas un préalable technique, c'est le cœur du sujet.** Sans lui, le chiffre
   le plus élémentaire est faux d'un ordre de grandeur. C'est la démonstration directe de l'utilité
   du projet, et ce sera la figure d'ouverture du mémoire.
2. **On peut s'appuyer sur le travail du producteur avant de faire le sien.** Sa colonne
   `montant_anomalie` suffit déjà à redresser les totaux. En phase 5, notre propre détection devra
   être comparée à la sienne, et non repartir de zéro.

Les années 2019 et 2020 restent basses pour une autre raison : la couverture des sources était
encore incomplète. Un creux dans une série temporelle n'est pas forcément un phénomène réel, il
peut n'être qu'un défaut de collecte. À vérifier avant toute interprétation.

## 3. La saisonnalité

La commande publique suit le calendrier budgétaire, pas le calendrier économique. C'est une
information utile pour le site : un acheteur qui publie en août n'a pas le même comportement que
celui qui publie en décembre.

In [5]:
con.sql("""
    select
        month(dateNotification) as mois,
        count(*)                as marches
    from marches
    where dateNotification between date '2019-01-01' and date '2025-12-31'
    group by 1
    order by 1
""").df()

,mois,marches
0,1,146566
1,2,124759
2,3,150417
3,4,139143
4,5,136491
5,6,171309
6,7,190971
7,8,112902
8,9,130591
9,10,161934


**Décembre et juillet sont les pics, août est le creux.** Décembre correspond à la clôture de
l'exercice budgétaire, juillet à la fin du premier semestre et à l'anticipation des congés.

Conséquence concrète pour la phase 5 : une détection d'anomalie qui ignorerait cette saisonnalité
signalerait chaque décembre comme anormal. Il faudra comparer un marché à ses pairs du même mois,
pas à la moyenne de l'année.

## 4. Qui achète, et où

In [6]:
con.sql("""
    select
        coalesce(acheteur_categorie, '(non renseigné)') as categorie,
        count(*)                                        as marches,
        round(median(montant), 0)                       as montant_median
    from marches
    group by 1
    order by marches desc
""").df()

,categorie,marches,montant_median
0,Commune,687383,103000.0
1,(non renseigné),401126,106400.0
2,Groupement de communes,357434,150000.0
3,Département,192538,200000.0
4,EPIC,152284,110000.0
5,Établissement hospitalier,121877,100000.0
6,Syndicat mixte,90266,200000.0
7,État,53697,111577.0
8,Région,53401,207525.0
9,Département outre-mer,4669,300000.0


**Les communes dominent en nombre**, avec environ 687 000 marchés, devant les groupements de
communes et les départements. Mais **401 126 marchés n'ont pas de catégorie d'acheteur renseignée**,
soit 22 %. Le champ existe, il est simplement vide.

C'est un problème d'enrichissement, pas de qualité au sens strict : l'identifiant de l'acheteur,
lui, est présent. L'API Recherche d'entreprises permettra de combler ce vide en phase 3. Première
justification concrète de cette troisième source de données.

In [7]:
con.sql("""
    select
        coalesce(acheteur_region_nom, '(non renseignée)') as region,
        count(*)                                          as marches
    from marches
    group by 1
    order by marches desc
    limit 8
""").df()

,region,marches
0,Île-de-France,307740
1,Auvergne-Rhône-Alpes,286804
2,Occitanie,224259
3,Provence-Alpes-Côte d'Azur,183543
4,Nouvelle-Aquitaine,180434
5,Grand Est,152884
6,Hauts-de-France,146099
7,Pays de la Loire,141545


## 5. Qu'achète-t-on

Le code CPV est la nomenclature européenne des achats publics. Ses deux premiers chiffres donnent
la division, soit la grande famille d'achat.

In [8]:
pd.read_csv("../mesures/resultats/familles-cpv.csv").head(10)

,famille_cpv,marches,montant_median
0,45,821967,146538.0
1,71,249815,105300.0
2,79,103818,92968.0
3,33,78628,82938.0
4,90,74867,167985.0
5,50,69351,125000.0
6,34,53844,103600.0
7,44,50766,112500.0
8,72,47604,123586.0
9,39,46704,90000.0


**La construction écrase tout** : environ 822 000 marchés pour la division 45, soit plus du triple
de la deuxième famille, les services d'architecture et d'ingénierie. Viennent ensuite les services
aux entreprises, le matériel médical et les déchets.

Conséquence pour la comparaison des prix : **comparer un marché à la moyenne générale n'a aucun
sens**. Un marché de voirie et un marché de logiciels n'ont rien de comparable. Tout écart devra se
mesurer à l'intérieur d'une même famille CPV, et si possible d'une même tranche de montant.

## 6. Un problème de normalisation, réglé sans modèle

Regardons comment est écrite la nature du marché.

In [9]:
con.sql("""
    select nature as ecriture, count(*) as marches
    from marches
    where nature is not null
    group by 1
    order by marches desc
""").df()

,ecriture,marches
0,Marché,1546910
1,Accord-cadre,228783
2,MARCHE,151851
3,ACCORD-CADRE,133538
4,Marché subséquent,26126
5,MARCHE SUBSEQUENT,17681
6,Marché de partenariat,607
7,Accord cadre,76
8,marché,42
9,Marché de défense ou de sécurité,30


Le même concept s'écrit **`Marché`, `MARCHE`, `MARCHÉ`, `marché`** selon la source. Douze écritures
pour six notions réelles. C'est le résultat direct de l'agrégation de 63 sources différentes, chacune
avec ses habitudes de saisie.

In [10]:
# Deux opérations suffisent : passer en majuscules et retirer les accents.
# Pas de modèle, pas de rapprochement flou, pas d'apprentissage : deux fonctions SQL.
con.sql("""
    select
        strip_accents(upper(nature)) as nature_normalisee,
        count(*)                     as marches,
        count(distinct nature)       as ecritures_regroupees
    from marches
    where nature is not null
    group by 1
    order by marches desc
""").df()

,nature_normalisee,marches,ecritures_regroupees
0,MARCHE,1698803,3
1,ACCORD-CADRE,362321,2
2,MARCHE SUBSEQUENT,43807,2
3,MARCHE DE PARTENARIAT,618,2
4,ACCORD CADRE,76,1
5,MARCHE DE DEFENSE OU DE SECURITE,30,1
6,CONCESSION DE SERVICE PUBLIC,1,1


**Douze écritures ramenées à sept lignes, avec deux fonctions SQL.**

Mais le résultat montre aussi la limite de la méthode : `ACCORD-CADRE` et `ACCORD CADRE` restent
séparés, parce qu'ils ne diffèrent que par un tiret. Normaliser la ponctuation réglerait ce cas,
et c'est encore une règle simple. La question à se poser à chaque fois est la même : la marche
suivante de l'escalier apporte-t-elle assez pour justifier son coût ?

C'est une illustration parfaite du principe de l'escalier qui guide tout le projet : règles et
expressions régulières, puis statistique, puis apprentissage automatique, puis embeddings, puis
grands modèles de langage. **On ne monte une marche que si la mesure prouve que la précédente ne
suffit pas.**

Ici, la marche la plus basse suffit et coûte deux millisecondes. Confier cette normalisation à un
modèle de langage serait plus lent, plus cher, non déterministe, et impossible à justifier devant un
jury. Le même raisonnement s'appliquera aux noms d'acheteurs et de titulaires, où les règles
simples ne suffiront probablement pas, et où il faudra effectivement monter d'une marche.

## 7. Les offres reçues : la difficulté principale du projet

In [11]:
con.sql("""
    select
        count(*)                                        as lignes_renseignees,
        count(distinct uid)                             as marches_renseignes,
        round(
            100.0 * count(distinct uid)
            / (select count(distinct uid) from marches), 1
        )                                               as part_des_marches,
        sum(case when offresRecues = 1 then 1 else 0 end)          as une_seule_offre,
        round(
            100.0 * sum(case when offresRecues = 1 then 1 else 0 end) / count(*), 1
        )                                                          as part_une_offre,
        round(median(offresRecues), 1)                             as mediane,
        sum(case when offresRecues > 100 then 1 else 0 end)        as plus_de_100_offres,
        max(offresRecues)                                          as maximum
    from marches
    where offresRecues is not null
""").df()

,lignes_renseignees,marches_renseignes,part_des_marches,une_seule_offre,part_une_offre,mediane,plus_de_100_offres,maximum
0,899298,800782,43.7,197581.0,22.0,3.0,13597.0,20300


L'information existe pour **800 782 marchés sur 1 833 468, soit 44 %**. Sur ce sous-ensemble,
la médiane est de **3 offres** et **22 % n'ont reçu qu'une seule offre**.

Attention au piège de comptage, qui vaut pour tout ce jeu de données : 899 298 est un nombre de
**lignes**, pas de marchés. Un marché attribué à trois entreprises occupe trois lignes. Diviser un
nombre de lignes par un nombre de marchés donne un pourcentage faux, et c'est exactement le genre
d'erreur qu'un jury repère.

Ce taux de 22 % est cohérent avec la littérature européenne sur le sujet, ce qui est plutôt
rassurant : le sous-ensemble renseigné ne semble pas complètement biaisé. Mais « semble » n'est pas
une preuve, et c'est précisément ce qu'il faudra mesurer.

Le maximum, **13 597 offres pour un seul marché**, confirme que même la partie renseignée contient
des saisies fausses.

**La décision à prendre avant la phase 5**, dans un ADR :

1. travailler sur les 44 % renseignés, en mesurant le biais de sélection, par exemple en comparant
   la répartition par catégorie d'acheteur, par famille CPV et par montant entre les deux
   sous-ensembles ;
2. croiser avec le BOAMP, qui publie les avis d'attribution et peut combler une partie du trou ;
3. changer d'indicateur, par exemple la concentration des attributions par acheteur, qui ne dépend
   pas de ce champ.

Ces trois pistes ne s'excluent pas. La troisième a un mérite particulier : elle est calculable sur
100 % des données.

## 8. Les durées, dernier champ à valider

In [12]:
con.sql("""
    select
        sum(case when dureeMois is null then 1 else 0 end) as absentes,
        round(median(dureeMois), 0)                        as mediane_mois,
        sum(case when dureeMois > 120 then 1 else 0 end)   as plus_de_10_ans,
        max(dureeMois)                                     as maximum_mois
    from marches
""").df()

,absentes,mediane_mois,plus_de_10_ans,maximum_mois
0,1425.0,24.0,6042.0,32000


Médiane à **24 mois**, ce qui est plausible. Mais **6 042 marchés dépassent dix ans**, et le maximum
atteint **32 000 mois, soit plus de 2 600 ans**.

Là encore, une règle simple suffit : une durée supérieure à un plafond raisonnable est marquée comme
invalide, sans être supprimée. **Marquer plutôt que supprimer** est un principe à tenir dans tout le
projet : une ligne effacée ne peut plus être expliquée, et l'administration qui l'a publiée ne peut
plus la corriger.

## 9. Ce qu'on retient, et ce que ça change pour la suite

| Constat | Conséquence |
|---|---|
| Total brut annuel faux d'un facteur 10 à 20 | le nettoyage est le cœur du sujet, pas un préalable technique |
| La colonne `montant_anomalie` du producteur suffit à redresser les totaux | s'appuyer dessus, puis comparer notre détection à la sienne |
| Médiane à 126 k€, distribution très asymétrique | raisonner en médiane et en quantiles, jamais en moyenne |
| Pics de décembre et de juillet | comparer un marché à ses pairs du même mois |
| La construction représente près de la moitié des marchés | comparer les prix à l'intérieur d'une même famille CPV |
| 22 % de catégories d'acheteur manquantes | enrichissement par l'API Recherche d'entreprises, phase 3 |
| 12 écritures pour 6 natures de marché | deux fonctions SQL suffisent : premier barreau de l'escalier |
| Offres reçues absentes dans 51 % des cas | ADR à écrire avant la phase 5 |
| Durées jusqu'à 2 600 ans | marquer, ne pas supprimer |

**Figures produites** : `make figures` génère les cinq figures dans `memoire/figures/`, depuis les
CSV de `mesures/resultats/`. Elles sont prêtes à être insérées dans le mémoire.